# Phase G - "Thay" emotion2vec+ day them "tro" EmoBranch F2 nhan cam xuc tren giong noi NGOAI MEAD

**Van de:** F2 chi hoc tren MEAD (dien vien, phong thu) -> gap giong ngoai doi nhan cam xuc chua ro.
**Cach lam (teacher - student, y tuong *Tiny is not small enough*):** phan NHAN CAM XUC cua F2 khong can landmark ->
hoc them tren du lieu chi co audio:

| Nguon | Nhan | Dung de |
| :--- | :--- | :--- |
| MEAD (speaker train) | that + landmark | giu nguyen kha nang bieu cam tinh (loss LMD) + nhan |
| **CREMA-D** (91 nguoi, ODbL) | that (NEU / HAP / ANG ...) + **xac suat cua thay** | da dang giong, giong it "dien" |
| **RAVDESS** (24 dien vien, CC BY-NC-SA) | that + xac suat cua thay | them giong, kiem tra cheo |
| `extra_audio/unlabeled/` (tuy chon) | **chi co thay gan nhan** (giu cua so thay chac >= `PSEUDO_CONF`) | audio that bat ky (podcast, video...) |

Loss cho mau ngoai MEAD = CE voi nhan that (neu co) + **KD** (KL voi xac suat mem cua thay, nhiet do T).
Chi so 3 lop cua F2 (neutral + cap da chon o Phase F2). Chon `best` theo **UAR trung binh tren val cua ca 3 nguon**,
voi dieu kien bieu cam tinh tren MEAD val khong xau qua `STATIC_TOL`.

**Luu tru:** audio CREMA-D / RAVDESS tai vao o dia Colab (`/content/extra`), **chi luu dac trung** len Drive
(`Data_emotion_mead_task_b/phaseG_extra/`, ~30-50 MB). Ket qua: `student_a2l/phaseF/phaseG_final.pth` (demo tu dung neu co).

| Cell | Viec | Thoi gian (T4) |
| :--- | :--- | :--- |
| 0 - 2 | tim Drive, cau hinh, cai dat | 2 phut |
| 3 | tai CREMA-D (chi 3 cam xuc can dung) + RAVDESS | 5 - 10 phut |
| 4 | trich dac trung + xac suat cua thay (tu chay tiep) | 10 - 15 phut, 1 lan |
| 5 | nap du lieu, chia nguoi noi, danh gia TRUOC khi hoc | 2 phut |
| 6 | hoc them (resume) | 5 - 10 phut |
| 7 | danh gia test: thay / F2 cu / F2 moi tren MEAD, CREMA-D, RAVDESS | 1 phut |
| 8 | xuat `phaseG_final.pth` | |

## Cell 0 - Mount Drive va tu tim thu muc du lieu

In [ ]:
# Cell 0 - Mount Drive + TU TIM thu muc du lieu (chay duoc ca khi dung tai khoan Google khac, co shortcut)
import os, sys, glob
from pathlib import Path

DRIVE_ROOT_OVERRIDE = ''      # neu tu tim sai: dien tay, vd '/content/drive/MyDrive/DAT301m/Data_student'
MEAD_ROOT_OVERRIDE  = ''      # vd '/content/drive/.shortcut-targets-by-id/1GwXP.../MEAD'

if 'google.colab' in sys.modules and not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')


def _is_data_root(p):
    # thu muc Data_student that: co du lieu MEAD da xu ly hoac thu muc ket qua student_a2l
    return os.path.isdir(os.path.join(p, 'Data_emotion_mead_task_b')) or os.path.isdir(os.path.join(p, 'student_a2l'))


def _is_mead(p):
    return os.path.isdir(p) and any(os.path.isfile(os.path.join(p, d, 'audio.tar')) for d in ('M003', 'M005', 'W033', 'M032'))


def _score(p):
    # 3: co du lieu MEAD da xu ly (split.json) | 2: co thu muc MEAD da xu ly | 1: chi co thu muc ket qua student_a2l
    if os.path.isfile(os.path.join(p, 'Data_emotion_mead_task_b', 'phaseB_ready', 'split.json')):
        return 3
    if os.path.isdir(os.path.join(p, 'Data_emotion_mead_task_b')):
        return 2
    return 1 if os.path.isdir(os.path.join(p, 'student_a2l')) else 0


def find_drive_root():
    if DRIVE_ROOT_OVERRIDE:
        return Path(DRIVE_ROOT_OVERRIDE)
    base = '/content/drive'
    cands = [base + '/MyDrive/DAT301m/Data_student']
    # shortcut nam o MyDrive (ten bat ky), shortcut tro toi DAT301m hoac Data_student, thu muc chia se (Shared drives)
    for pat in ('/MyDrive/*/Data_student', '/MyDrive/*/*/Data_student', '/MyDrive/Data_student',
                '/.shortcut-targets-by-id/*/DAT301m/Data_student', '/.shortcut-targets-by-id/*/Data_student',
                '/.shortcut-targets-by-id/*/*/Data_student', '/.shortcut-targets-by-id/*/*/*/Data_student',
                '/Shareddrives/*/DAT301m/Data_student', '/Shareddrives/*/Data_student', '/MyDrive/*'):
        cands += sorted(glob.glob(base + pat))
    seen, scored = set(), []
    for c in cands:
        r = os.path.realpath(c)
        if r in seen or not os.path.isdir(c):
            continue
        seen.add(r)
        sc = _score(c)
        if sc:
            scored.append((sc, c))
    for sc, c in sorted(scored, key=lambda x: -x[0]):
        print('  ung vien: %-90s diem %d' % (c, sc))
    best = max(scored, key=lambda x: x[0]) if scored else None     # max giu thu tu uu tien khi bang diem
    return Path(best[1]) if best else None


def find_mead_root():
    if MEAD_ROOT_OVERRIDE:
        return Path(MEAD_ROOT_OVERRIDE)
    base = '/content/drive'
    for pat in ('/.shortcut-targets-by-id/1GwXP-KpWOxOenOxITTsURJZQ_1pkd4-j/MEAD', '/.shortcut-targets-by-id/*/MEAD',
                '/MyDrive/MEAD', '/MyDrive/*/MEAD', '/MyDrive/*/*/MEAD', '/Shareddrives/*/MEAD', '/Shareddrives/*/*/MEAD'):
        for c in sorted(glob.glob(base + pat)):
            if _is_mead(c):
                return Path(c)
    return None


def _writable(p):
    try:
        t = Path(p) / '.write_test'
        t.write_text('ok'); t.unlink()
        return True
    except Exception:
        return False


if 'google.colab' in sys.modules:
    DRIVE_FOUND, MEAD_FOUND = find_drive_root(), find_mead_root()
    assert DRIVE_FOUND is not None, ('Khong tim thay thu muc Data_student. Kiem tra shortcut (chuot phai thu muc DAT301m '
                                     'duoc chia se > Organize > Add shortcut > My Drive) hoac dien DRIVE_ROOT_OVERRIDE.')
    print('Data_student :', DRIVE_FOUND, '| ghi duoc:', _writable(DRIVE_FOUND))
    if _score(str(DRIVE_FOUND)) < 3:
        print('!!! Thu muc nay KHONG co Data_emotion_mead_task_b/phaseB_ready/split.json -> chua thay shortcut toi thu muc '
              'du lieu that. Them shortcut vao My Drive hoac dien DRIVE_ROOT_OVERRIDE.')
    print('MEAD goc     :', MEAD_FOUND if MEAD_FOUND else 'KHONG THAY (chi can cho cac cell doc audio MEAD goc)')
    if not _writable(DRIVE_FOUND):
        print('!!! Tai khoan nay chi co quyen XEM thu muc chia se -> khong luu duoc ket qua. '
              'Nho chu thu muc cap quyen "Nguoi chinh sua" (Editor) cho tai khoan nay.')

Mounted at /content/drive
  ung vien: /content/drive/MyDrive/DAT301m/Data_student                                                diem 3
Data_student : /content/drive/MyDrive/DAT301m/Data_student | ghi duoc: True
MEAD goc     : /content/drive/.shortcut-targets-by-id/1GwXP-KpWOxOenOxITTsURJZQ_1pkd4-j/MEAD


## Cell 1 - Cau hinh

In [ ]:
import os, sys, json, time, math, shutil, subprocess, zipfile, glob
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F

IN_COLAB = 'google.colab' in sys.modules
DRIVE    = Path('/content/drive/MyDrive/DAT301m/Data_student')
DRIVE    = globals().get('DRIVE_FOUND') or DRIVE
LIB_DIR, EXTRA = Path('/content'), Path('/content/extra')
_T = os.environ.get('PHASEG_TEST_ROOT')                 # chi dung khi chay thu ngoai Colab
if _T:
    DRIVE, LIB_DIR, EXTRA = Path(_T) / 'drive', Path(_T), Path(_T) / 'extra'
READY    = DRIVE / 'Data_emotion_mead_task_b' / 'phaseB_ready'
E2V_DIR  = DRIVE / 'Data_emotion_mead_task_b' / 'phaseF_e2v'
G_DIR    = DRIVE / 'Data_emotion_mead_task_b' / 'phaseG_extra'       # dac trung audio ngoai MEAD
UNLAB_DIR = DRIVE / 'extra_audio' / 'unlabeled'                       # (tuy chon) tha audio bat ky vao day
OUT      = DRIVE / 'student_a2l' / 'phaseF'
F2_CKPT  = OUT / 'phaseF2_final.pth'

USE_CREMA, USE_RAVDESS, USE_UNLABELED = True, True, True
CREMA_URLS = ['https://gitlab.com/cs-cooper-lab/crema-d-mirror.git', 'https://github.com/CheyneyComputerScience/CREMA-D.git']
RAVDESS_URL = 'https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1'
CREMA_MAP = {'NEU': 'neutral', 'HAP': 'happy', 'ANG': 'angry', 'SAD': 'sad', 'FEA': 'fear', 'DIS': 'disgusted'}
RAV_MAP = {'01': 'neutral', '03': 'happy', '04': 'sad', '05': 'angry', '06': 'fear', '07': 'disgusted', '08': 'surprised'}
UNLAB_WIN = 3.0                    # audio khong nhan: cat cua so 3 s
PSEUDO_CONF = 0.6                  # chi giu cua so thay chac >= 0.6 (tren 3 lop)

CFG = dict(epochs=30, batch_size=256, lr=3e-4, weight_decay=1e-2, warmup_epochs=1, steps_per_epoch=150,
           mead_frac=0.5,           # 50% moi batch la MEAD (giu bieu cam tinh), 50% audio ngoai
           kd_T=2.0, lambda_kd=1.0, lambda_ce_ext=0.5, lambda_static=1.0, lambda_cls=0.2, lambda_lvl=0.1,
           label_dropout=0.5, crop_p=0.5, crop_min_sec=2.0, label_smooth=0.1, seed=0)
STATIC_TOL = 0.03                  # chap nhan val LMD bieu cam tinh (MEAD) xau di toi da 3%
if os.environ.get('PHASEG_SMOKE'):
    CFG.update(epochs=3, steps_per_epoch=20)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

device: cuda


## Cell 2 - Cai dat, ghi thu vien, nap F2

In [ ]:
LIB_SRC = r'''"""Phase F - hoc cam xuc TU CHINH AUDIO bang dac trung emotion2vec+ (dong bang).

* E2VFeat: chay emotion2vec+ -> dac trung tung khung (~50 khung/giay, 1024 chieu ban large) + xac suat 8 nhan.
  Utterance embedding cua emotion2vec+ = TRUNG BINH cac khung (xem funasr/models/emotion2vec/model.py), nen chi can luu
  trung binh theo tung doan CHUNK giay + so khung cua doan: ghep lai duoc embedding cua bat ky cua so nao (tiet kiem
  luu tru ~50 lan so voi luu tung khung).
* EmoBranch: MLP nho [embedding emotion2vec+ | nhan (tuy chon)] -> (a) bieu cam tinh TOAN MAT (136), (b) logits 8 cam xuc,
  (c) muc 1..3. Vector dieu kien cho mo hinh chuyen dong C4 = xac suat 7 cam xuc x muc / 3 (dung dinh dang one-hot x muc / 3
  ma C4 da hoc) -> khong can train lai C4.
* LandmarkEmoClassifier: phan loai cam xuc tu bieu cam tinh (136) -> loss nhat quan cam xuc (kieu EMOTE) va metric Emo-Acc.
"""
import numpy as np

EMO_NAMES = ['neutral', 'angry', 'disgusted', 'fear', 'happy', 'sad', 'surprised', 'contempt']
E2V_MAP = {'angry': 'angry', 'disgusted': 'disgusted', 'fearful': 'fear', 'happy': 'happy', 'neutral': 'neutral',
           'sad': 'sad', 'surprised': 'surprised', 'other': 'neutral', 'unknown': 'neutral'}


# ============================================================================= emotion2vec+ -> dac trung
def _e2v_label(lab):
    return E2V_MAP.get(str(lab).split('/')[-1].strip().lower())


class E2VFeat:
    """emotion2vec+ (FunASR), lay dac trung TUNG KHUNG + xac suat. hub 'hf' loi thi thu 'ms' (ModelScope)."""

    def __init__(self, size='large', device='cuda', hub='hf'):
        from funasr import AutoModel
        err = None
        for h in ([hub] + [x for x in ('hf', 'ms') if x != hub]):
            name = ('emotion2vec/emotion2vec_plus_%s' if h == 'hf' else 'iic/emotion2vec_plus_%s') % size
            try:
                self.m = AutoModel(model=name, hub=h, device=device, disable_update=True, disable_pbar=True,
                                   disable_log=True)
                break
            except Exception as e:
                err = e
        else:
            raise RuntimeError('Khong tai duoc emotion2vec+: %r' % err)
        self.size = size
        self.head = self._get_head()

    def _get_head(self):
        """Lop phan loai cua emotion2vec+ (Linear tren trung binh khung) -> tinh xac suat cho cua so bat ky."""
        try:
            mdl = self.m.model
            W = mdl.proj.weight.detach().float().cpu().numpy(); b = mdl.proj.bias.detach().float().cpu().numpy()
            kw = self.m.kwargs
            labels = list(getattr(kw.get('tokenizer'), 'token_list', None) or kw.get('token_list'))
            assert len(labels) == W.shape[0]
            return dict(W=W, b=b, labels=np.array([str(x) for x in labels]))
        except Exception as e:                                            # khong lay duoc -> van chay, chi mat tien ich
            print('[canh bao] khong lay duoc lop phan loai emotion2vec+:', repr(e)[:200])
            return None

    def frames(self, y16k):
        """audio float32 16 kHz -> (dac trung (T, D) float32, xac suat (8,) theo EMO_NAMES)."""
        r = self.m.generate(np.asarray(y16k, np.float32), granularity='frame', extract_embedding=True,
                            disable_pbar=True)[0]
        f = np.asarray(r['feats'], np.float32)
        if f.ndim == 1:
            f = f[None]
        return f, scores_to8(r['labels'], r['scores'])


def frames_long(e2v, y16k, chunk_sec=20.0):
    """Audio dai (vai phut) -> dac trung khung, chay tung doan chunk_sec giay (tranh het bo nho GPU)."""
    n = int(chunk_sec * 16000); out = []
    for s in range(0, max(len(y16k), 1), n):
        seg = y16k[s:s + n]
        if len(seg) < 1600 and out:                 # mau cuoi qua ngan -> lap lai khung cuoi
            out.append(out[-1][-1:]); continue
        out.append(e2v.frames(seg)[0])
    return np.concatenate(out)


def scores_to8(labels, scores):
    p = np.zeros(8, np.float32)
    for lab, sc in zip(labels, scores):
        k = _e2v_label(lab)
        if k is not None:
            p[EMO_NAMES.index(k)] += float(sc)
    return p / max(p.sum(), 1e-8)


def head_probs(head, emb):
    """emb (N, D) trung binh khung -> xac suat (N, 8) bang lop phan loai goc cua emotion2vec+."""
    emb = np.atleast_2d(np.asarray(emb, np.float32))
    lg = emb @ head['W'].T + head['b']
    keep = np.array([not str(l).startswith('unuse') for l in head['labels']])
    lg[:, ~keep] = -np.inf
    lg = lg - lg.max(1, keepdims=True); p = np.exp(lg); p /= p.sum(1, keepdims=True)
    out = np.zeros((len(emb), 8), np.float32)
    for j, lab in enumerate(head['labels']):
        k = _e2v_label(lab)
        if keep[j] and k is not None:
            out[:, EMO_NAMES.index(k)] += p[:, j]
    return out / np.maximum(out.sum(1, keepdims=True), 1e-8)


def chunk_means(feats, dur_sec, chunk_sec):
    """dac trung khung (T, D) -> (trung binh tung doan chunk_sec giay (K, D), so khung moi doan (K,)).
    chunk_sec <= 0: chi 1 doan = ca clip."""
    T = len(feats)
    if chunk_sec <= 0 or dur_sec <= chunk_sec:
        return feats.mean(0, keepdims=True), np.array([T], np.int32)
    fps = T / max(dur_sec, 1e-6)
    edges = np.unique(np.clip(np.round(np.arange(0, dur_sec + 1e-9, chunk_sec) * fps).astype(int), 0, T))
    if edges[-1] != T:
        edges = np.append(edges, T)
    if T - edges[-2] < 0.4 * chunk_sec * fps and len(edges) > 2:        # doan cuoi qua ngan -> gop vao doan truoc
        edges = np.delete(edges, -2)
    M = np.stack([feats[a:b].mean(0) for a, b in zip(edges[:-1], edges[1:])])
    return M, np.diff(edges).astype(np.int32)


def pool(chunks, counts, a=0, b=None):
    """trung binh co trong so cua cac doan [a, b) = embedding cua cua so do."""
    c = counts[a:b].astype(np.float32)
    return (chunks[a:b].astype(np.float32) * c[:, None]).sum(0) / max(c.sum(), 1.0)


def window_embeddings(feats, dur_sec, win=2.5, hop=0.5):
    """Cho audio dai (demo): dac trung khung -> (tam cua so giay (N,), embedding (N, D)). Cua so win giay, buoc hop."""
    T = len(feats); fps = T / max(dur_sec, 1e-6)
    n, h = max(1, int(round(win * fps))), max(1, int(round(hop * fps)))
    st = list(range(0, max(T - n, 0) + 1, h)) or [0]
    if st[-1] != max(T - n, 0):
        st.append(max(T - n, 0))
    cs = np.concatenate([np.zeros((1, feats.shape[1])), np.cumsum(feats, 0, dtype=np.float64)])
    E = np.stack([(cs[min(s + n, T)] - cs[s]) / (min(s + n, T) - s) for s in st]).astype(np.float32)
    return (np.array(st) + min(n, T) / 2) / fps, E


# ============================================================================= mo hinh
def build_emo_branch(in_dim=1024, label_dim=7, hidden=256, out_dim=136, drop=0.3, feat_drop=0.1):
    import torch
    import torch.nn as nn

    class EmoBranch(nn.Module):
        """[emb emotion2vec+ (D) | nhan 7 chieu x muc/3 | co nhan?] -> (tinh 136, logits 8, muc 1..3)."""

        def __init__(self):
            super().__init__()
            self.cfg = dict(in_dim=in_dim, label_dim=label_dim, hidden=hidden, out_dim=out_dim, drop=drop,
                            feat_drop=feat_drop)
            self.norm = nn.LayerNorm(in_dim)
            self.fdrop = nn.Dropout(feat_drop)
            self.body = nn.Sequential(nn.Linear(in_dim + label_dim + 1, hidden), nn.GELU(), nn.Dropout(drop),
                                      nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(drop))
            self.static = nn.Linear(hidden, out_dim)
            self.lin = nn.Linear(in_dim, out_dim, bias=False)     # duong tat tuyen tinh (bao gom duoc ca ridge)
            self.cls = nn.Linear(hidden, 8)
            self.lvl = nn.Linear(hidden, 1)
            for p in (self.static.weight, self.static.bias, self.lin.weight):
                nn.init.zeros_(p)

        def forward(self, emb, label_vec=None, has_label=None):
            # label_vec (B, 7) = one-hot x muc/3 (neutral = 0); has_label (B,) 1 = co nhan, 0 = chi dung audio.
            # Can co has_label rieng vi neutral co nhan cung la vector 0.
            x = self.fdrop(self.norm(emb))
            if label_vec is None:
                lab = x.new_zeros(len(x), self.cfg['label_dim'] + 1)
            else:
                has = (x.new_ones(len(x)) if has_label is None else has_label.to(x.dtype)).reshape(-1, 1)
                lab = torch.cat([label_vec.to(x.dtype) * has, has], 1)
            h = self.body(torch.cat([x, lab], 1))
            return self.static(h) + self.lin(x), self.cls(h), 1.0 + 2.0 * torch.sigmoid(self.lvl(h)[:, 0])

        @staticmethod
        def cond_vector(logits, lvl):
            """-> vector 7 chieu cho mo hinh chuyen dong C4 (xac suat cam xuc x muc / 3; neutral = 0)."""
            p = torch.softmax(logits.float(), -1)
            return p[:, 1:] * (lvl.float()[:, None] / 3.0)

    return EmoBranch()


def build_lm_classifier(in_dim=136, hidden=128, drop=0.3):
    import torch.nn as nn
    m = nn.Sequential(nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(drop), nn.Linear(hidden, hidden), nn.GELU(),
                      nn.Dropout(drop), nn.Linear(hidden, 8))
    m.cfg = dict(in_dim=in_dim, hidden=hidden, drop=drop)
    return m


def load_emo_branch(path, device='cpu'):
    """Nap phaseF_final.pth -> (EmoBranch, checkpoint dict: chua 'e2v_head', 'chunk_sec', 'lm_scale', ...)."""
    import torch
    ck = torch.load(str(path), map_location='cpu', weights_only=False)
    m = build_emo_branch(**ck['branch_cfg'])
    m.load_state_dict(ck['branch_state'])
    return m.to(device).eval(), ck
'''
(LIB_DIR / 'emo_audio_lib.py').write_text(LIB_SRC)
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))
import importlib, emo_audio_lib as EA
importlib.reload(EA)
if IN_COLAB:
    try:
        import funasr  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'funasr', 'modelscope'], check=False)
assert F2_CKPT.exists(), 'Thieu %s -> chay notebook phaseF (Cell 9-11) truoc' % F2_CKPT
F2M = torch.load(str(F2_CKPT), map_location='cpu', weights_only=False)
KEEP, KEEP_NAMES = list(F2M['keep']), tuple(F2M['keep_names'])
CLS = ['neutral'] + list(KEEP_NAMES)                    # 3 lop
EMO_NAMES = EA.EMO_NAMES
CHUNK_SEC, ZERO_LMD = float(F2M['chunk_sec']), float(F2M['zero_lmd'])
SPLIT = json.loads((READY / 'split.json').read_text())
G_DIR.mkdir(parents=True, exist_ok=True); EXTRA.mkdir(parents=True, exist_ok=True)
print('Lop:', CLS, '| chunk', CHUNK_SEC, 's | e2v', F2M['e2v_size'])

Lop: ['neutral', 'angry', 'happy'] | chunk 1.0 s | e2v large


## Cell 3 - Tai CREMA-D (chi 3 cam xuc can dung) va RAVDESS vao o dia Colab

CREMA-D dung git-lfs: chi keo cac file `AudioWAV/*_NEU_*`, `*_HAP_*`, `*_ANG_*` (theo lop dang giu), khong keo video (~7 GB).
Bi loi mang thi co the tu tai roi dat vao Drive `Data_student/extra_audio/CREMA-D/AudioWAV/` va
`Data_student/extra_audio/RAVDESS/Actor_xx/` - cell nay se dung ban tren Drive.

In [ ]:
CREMA_DIR, RAV_DIR = EXTRA / 'crema' / 'AudioWAV', EXTRA / 'ravdess'
codes = [k for k, v in CREMA_MAP.items() if v in CLS]


def real_wavs(d):
    return [p for p in Path(d).glob('*.wav') if p.stat().st_size > 2000]       # bo file con tro git-lfs


def feats_done(name):
    return (G_DIR / (name + '.npz')).exists()


if USE_CREMA and not feats_done('crema') and len(real_wavs(CREMA_DIR)) < 100:
    dr = DRIVE / 'extra_audio' / 'CREMA-D' / 'AudioWAV'
    if len(real_wavs(dr)) >= 100:
        CREMA_DIR = dr
    else:
        subprocess.run('apt-get -qq install -y git-lfs >/dev/null 2>&1; git lfs install >/dev/null 2>&1', shell=True)
        inc = ','.join('AudioWAV/*_%s_*' % c for c in codes)
        for url in CREMA_URLS:
            shutil.rmtree(EXTRA / 'crema', ignore_errors=True)
            r = subprocess.run('GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 %s %s && cd %s && git lfs pull --include "%s"'
                               % (url, EXTRA / 'crema', EXTRA / 'crema', inc), shell=True, capture_output=True, text=True)
            n = len(real_wavs(CREMA_DIR))
            print('CREMA-D tu %s: %d file wav %s' % (url, n, '' if n else r.stderr[-300:]))
            if n >= 100:
                break
if USE_RAVDESS and not feats_done('ravdess') and len(list(RAV_DIR.glob('Actor_*/*.wav'))) < 100:
    dr = DRIVE / 'extra_audio' / 'RAVDESS'
    if len(list(dr.glob('Actor_*/*.wav'))) >= 100:
        RAV_DIR = dr
    else:
        import urllib.request
        zp = EXTRA / 'ravdess.zip'
        try:
            if not zp.exists():
                urllib.request.urlretrieve(RAVDESS_URL, zp)
            with zipfile.ZipFile(zp) as z:
                z.extractall(RAV_DIR)
            zp.unlink()
        except Exception as e:
            print('Khong tai duoc RAVDESS:', repr(e)[:200])
print('CREMA-D: %d wav | RAVDESS: %d wav | khong nhan: %d file'
      % (len(real_wavs(CREMA_DIR)), len(list(RAV_DIR.glob('Actor_*/*.wav'))),
         len([p for p in UNLAB_DIR.glob('**/*') if p.suffix.lower() in ('.wav', '.mp3', '.m4a', '.flac', '.ogg', '.mp4')])
         if UNLAB_DIR.exists() else 0))

CREMA-D tu https://gitlab.com/cs-cooper-lab/crema-d-mirror.git: 3629 file wav 
Khong tai duoc RAVDESS: ContentTooShortError('retrieval incomplete: got only 85027188 out of 208468073 bytes')
CREMA-D: 3629 wav | RAVDESS: 0 wav | khong nhan: 0 file


## Cell 4 - Trich dac trung emotion2vec+ + xac suat cua THAY (chi chay 1 lan, tu chay tiep)

Moi nguon -> `phaseG_extra/<nguon>.npz`: trung binh moi `CHUNK_SEC` giay (giong Phase F), xac suat 8 cam xuc cua
emotion2vec+ (= nhan mem cua thay), nhan that (neu co), nguoi noi. Audio khong nhan duoc cat cua so `UNLAB_WIN` giay.

In [ ]:
def items_crema():
    out = []
    for p in sorted(real_wavs(CREMA_DIR)):
        q = p.stem.split('_')
        if len(q) >= 3 and CREMA_MAP.get(q[2]) in CLS:
            out.append((p, CREMA_MAP[q[2]], 'C' + q[0]))
    return out


def items_ravdess():
    out = []
    for p in sorted(RAV_DIR.glob('Actor_*/*.wav')):
        q = p.stem.split('-')
        if len(q) == 7 and q[0] == '03' and q[1] == '01' and RAV_MAP.get(q[2]) in CLS:
            out.append((p, RAV_MAP[q[2]], 'R' + q[6]))
    return out


def items_unlabeled():
    if not UNLAB_DIR.exists():
        return []
    fs = sorted(p for p in UNLAB_DIR.glob('**/*') if p.suffix.lower() in ('.wav', '.mp3', '.m4a', '.flac', '.ogg', '.mp4'))
    return [(p, None, 'U%03d' % i) for i, p in enumerate(fs)]


def load16k(p):
    r = subprocess.run(['ffmpeg', '-loglevel', 'error', '-i', str(p), '-f', 'f32le', '-ac', '1', '-ar', '16000', '-'], capture_output=True)
    return np.frombuffer(r.stdout, np.float32).copy() if r.returncode == 0 else np.zeros(0, np.float32)


def extract(name, items, window=None):
    f = G_DIR / (name + '.npz')
    if f.exists() or not items:
        return
    global E2V
    if 'E2V' not in globals():
        E2V = __import__('fake_e2v').FakeE2V() if os.environ.get('PHASEG_FAKE_E2V') else EA.E2VFeat(size=F2M['e2v_size'], device=DEVICE.type)
    CH, CNT, COFF, P8, LAB, SPK, NM = [], [], [0], [], [], [], []
    t0 = time.time()
    for i, (p, lab, spk) in enumerate(items):
        y = load16k(p)
        segs = [y] if window is None else [y[s:s + int(window * 16000)] for s in range(0, max(len(y) - int(1.5 * 16000), 1), int(window * 16000))]
        for k, seg in enumerate(segs):
            if len(seg) < 8000:                                       # < 0.5 s
                continue
            fr, p8 = E2V.frames(seg)
            ch, cnt = EA.chunk_means(fr, len(seg) / 16000.0, CHUNK_SEC)
            CH.append(ch); CNT.append(cnt); COFF.append(COFF[-1] + len(ch)); P8.append(p8)
            LAB.append(-1 if lab is None else EMO_NAMES.index(lab)); SPK.append(spk); NM.append('%s#%d' % (p.name, k))
        if (i + 1) % 500 == 0:
            print('   %s: %d/%d file | %.0f s' % (name, i + 1, len(items), time.time() - t0))
    np.savez(f, chunks=np.concatenate(CH).astype(np.float16), counts=np.concatenate(CNT).astype(np.int16),
             coff=np.array(COFF, np.int32), probs8=np.stack(P8).astype(np.float16), label=np.array(LAB, np.int8),
             spk=np.array(SPK), names=np.array(NM), chunk_sec=np.float32(CHUNK_SEC))
    print('%s: %d mau | %.1f MB | %.0f s' % (name, len(NM), f.stat().st_size / 1e6, time.time() - t0))


if USE_CREMA:
    extract('crema', items_crema())
if USE_RAVDESS:
    extract('ravdess', items_ravdess())
if USE_UNLABELED:
    extract('unlabeled', items_unlabeled(), window=UNLAB_WIN)
print('Dac trung:', {f.stem: '%.1f MB' % (f.stat().st_size / 1e6) for f in G_DIR.glob('*.npz')})

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


funasr version: 1.4.16.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.0.0.weight, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_large/snapshots/6c303ba987b86b93193de93e34bb2b077a6bedc4/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.0.0.bias, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_large/snapshots/6c303ba987b86b93193de93e34bb2b077a6bedc4/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.1.0.weight, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_large/snapshots/6c303ba987b86b93193de93e34bb2b077a6bedc4/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.1.0.bias, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_large/snapshots/6c303ba987b86b93193de93e34bb2b077a6bedc4/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.2.0.weight, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_large/snapshots

## Cell 5 - Nap du lieu, chia nguoi noi, danh gia TRUOC khi hoc

* MEAD: speaker train / val / test nhu Phase F (chi clip 3 lop). Muc tieu bieu cam tinh = TB landmark clip.
* CREMA-D, RAVDESS: chia **theo nguoi noi** (seed 0): 70% train / 10% val / 20% test.
* Khong nhan: chi dung de train, giu cua so ma thay chac >= `PSEUDO_CONF` (nhan gia = lop thay chon).
* Nhan mem cua thay = xac suat emotion2vec+ chi lay 3 lop roi chuan hoa lai.

In [ ]:
W, LEAD = 18, 8


def emo_vec(e, l):
    v = np.zeros(7, np.float32)
    if e > 0:
        v[e - 1] = l / 3.0
    return v


def pool_all(ch, cnt, coff):
    return np.stack([EA.pool(ch, cnt, coff[i], coff[i + 1]) for i in range(len(coff) - 1)])


def teacher3(p8):
    p = np.asarray(p8, np.float32)[:, KEEP]
    return p / np.maximum(p.sum(1, keepdims=True), 1e-6)


def load_mead(speakers, name):
    Y, EMO, LVL, CH, CNT, COFF, P8 = [], [], [], [], [], [0], []
    for sp in speakers:
        with np.load(READY / (sp + '.npz')) as z:
            raw, off, emo_, lvl_ = z['lm_res_raw'], z['offsets'], z['emo'], z['lvl']
        with np.load(E2V_DIR / (sp + '.npz')) as f:
            chunks, counts, coff, has, probs8 = f['chunks'], f['counts'], f['coff'], f['has'], f['probs8']
        for i in range(len(off) - 1):
            a, b = int(off[i]), int(off[i + 1])
            if (b - a) - LEAD < W or not has[i] or int(emo_[i]) not in KEEP:
                continue
            Y.append(raw[a:b - LEAD].astype(np.float32).mean(0)); EMO.append(int(emo_[i])); LVL.append(int(lvl_[i]))
            c0, c1 = int(coff[i]), int(coff[i + 1])
            CH.append(chunks[c0:c1].copy()); CNT.append(counts[c0:c1].copy()); COFF.append(COFF[-1] + c1 - c0)
            P8.append(probs8[i].astype(np.float32))
        del raw, chunks
    D = dict(name=name, Y=np.stack(Y), emo=np.array(EMO), lvl=np.array(LVL), chunks=np.concatenate(CH),
             counts=np.concatenate(CNT).astype(np.float32), coff=np.array(COFF), p3=teacher3(np.stack(P8)))
    D['lab'] = np.stack([emo_vec(e, l) for e, l in zip(D['emo'], D['lvl'])])
    D['full'] = pool_all(D['chunks'], D['counts'], D['coff'])
    D['y3'] = np.array([KEEP.index(e) for e in D['emo']])
    print('MEAD %-5s | %5d clip' % (name, len(Y)))
    return D


def load_extra(name):
    f = G_DIR / (name + '.npz')
    if not f.exists():
        return None
    with np.load(f) as z:
        D = {k: z[k] for k in ('chunks', 'counts', 'coff', 'probs8', 'label', 'spk', 'names')}
    D['counts'] = D['counts'].astype(np.float32)
    D['full'] = pool_all(D['chunks'], D['counts'], D['coff'])
    D['p3'] = teacher3(D['probs8'])
    D['y3'] = np.array([KEEP.index(int(l)) if int(l) in KEEP else -1 for l in D['label']])
    return D


def take(D, idx):
    S = {k: v[idx] for k, v in D.items() if isinstance(v, np.ndarray) and k not in ('chunks', 'counts', 'coff')
         and len(v) == len(D['full'])}
    S['row'] = np.asarray(idx); S['chunks'], S['counts'], S['coff'] = D['chunks'], D['counts'], D['coff']
    return S


def split_by_speaker(D, name, seed=0):
    spk = sorted(set(D['spk'].tolist())); rng = np.random.default_rng(seed); rng.shuffle(spk)
    n = len(spk); te = set(spk[:max(1, int(0.2 * n))]); va = set(spk[len(te):len(te) + max(1, int(0.1 * n))])
    part = lambda S: np.where(np.isin(D['spk'], list(S)) & (D['y3'] >= 0))[0]
    tr = np.where(~np.isin(D['spk'], list(te | va)) & (D['y3'] >= 0))[0]
    out = dict(train=take(D, tr), val=take(D, part(va)), test=take(D, part(te)))
    print('%-8s | nguoi noi %d (test %d, val %d) | mau train %d val %d test %d'
          % (name, n, len(te), len(va), len(tr), len(out['val']['y3']), len(out['test']['y3'])))
    return out


MTR, MVA, MTE = load_mead(SPLIT['train'], 'train'), load_mead(SPLIT['val'], 'val'), load_mead(SPLIT['test'], 'test')
EXT = {}
for nm in ('crema', 'ravdess'):
    D = load_extra(nm)
    if D is not None and (D['y3'] >= 0).sum() > 50:
        EXT[nm] = split_by_speaker(D, nm)
UNL = None
D = load_extra('unlabeled')
if D is not None:
    keep_ = np.where(D['p3'].max(1) >= PSEUDO_CONF)[0]
    if len(keep_):
        UNL = take(D, keep_); UNL['y3'] = UNL['p3'].argmax(1)
    print('khong nhan | %d cua so, giu %d (thay chac >= %.1f) | nhan gia: %s'
          % (len(D['full']), len(keep_), PSEUDO_CONF, dict(zip(CLS, np.bincount(UNL['y3'], minlength=3).tolist())) if UNL else {}))

# ---- tro (F2) ----
br = EA.build_emo_branch(**F2M['branch_cfg']).to(DEVICE)
br.load_state_dict(F2M['branch_state'])
KIDX = torch.tensor(KEEP, device=DEVICE)


@torch.no_grad()
def student(Dset, with_label=False):
    br.eval()
    E = torch.tensor(Dset['full']).to(DEVICE)
    lab = torch.tensor(Dset['lab']).to(DEVICE) if 'lab' in Dset else torch.zeros(len(E), 7, device=DEVICE)
    S, L, V = br(E, lab, torch.full((len(E),), 1.0 if with_label else 0.0, device=DEVICE))
    return S.cpu().numpy() * ZERO_LMD, torch.softmax(L[:, KIDX], -1).cpu().numpy()


def uar(y, p):
    return float(np.mean([(p[y == k] == k).mean() for k in np.unique(y)]))


def static_lmd(P, Y):
    return float(np.linalg.norm((P - Y).reshape(len(Y), 68, 2), axis=2).mean())


def evaluate(part):
    # -> {nguon: dict(uar_tro, uar_thay, recall_tro)} + LMD bieu cam tinh MEAD
    sets = {'MEAD': {'val': MVA, 'test': MTE}[part]}
    sets.update({k: v[part] for k, v in EXT.items()})
    out = {}
    for nm, Dset in sets.items():
        if len(Dset['y3']) == 0:
            continue
        S, P = student(Dset)
        pr, pt = P.argmax(1), Dset['p3'].argmax(1)
        out[nm] = dict(uar_tro=uar(Dset['y3'], pr), uar_thay=uar(Dset['y3'], pt), n=int(len(pr)),
                       recall_tro={CLS[k]: round(float((pr[Dset['y3'] == k] == k).mean()), 3) for k in np.unique(Dset['y3'])})
        if nm == 'MEAD':
            out[nm]['lmd_static'] = static_lmd(S, Dset['Y'])
    return out


def show_eval(ev, title):
    print('\n' + title)
    print('  %-9s %-6s %-12s %-12s %s' % ('nguon', 'mau', 'UAR tro(F2)', 'UAR thay', 'recall tro ' + '/'.join(CLS)))
    for nm, r in ev.items():
        print('  %-9s %-6d %-12.3f %-12.3f %s %s' % (nm, r['n'], r['uar_tro'], r['uar_thay'], '/'.join('%.2f' % r['recall_tro'].get(c, float('nan')) for c in CLS),
                                                  '| LMD tinh %.4f' % r['lmd_static'] if 'lmd_static' in r else ''))


EV0_VAL, EV0_TEST = evaluate('val'), evaluate('test')
show_eval(EV0_TEST, 'TRUOC khi hoc them (test):')

MEAD train |  7117 clip
MEAD val   |   875 clip
MEAD test  |   865 clip
crema    | nguoi noi 91 (test 18, val 9) | mau train 2553 val 359 test 717
ravdess  | nguoi noi 24 (test 4, val 2) | mau train 360 val 40 test 80

TRUOC khi hoc them (test):
  nguon     mau    UAR tro(F2)  UAR thay     recall tro neutral/angry/happy
  MEAD      865    0.949        0.950        0.93/0.95/0.97 | LMD tinh 0.0292
  crema     717    0.932        0.944        0.91/0.94/0.94 
  ravdess   80     0.979        0.990        1.00/1.00/0.94 


## Cell 6 - Hoc them: moi batch = 50% MEAD (giu bieu cam tinh) + 50% audio ngoai (nhan that + nhan mem cua thay)

* MEAD: loss nhu Phase F2 (LMD bieu cam tinh + phan loai + muc), co label dropout.
* Ngoai MEAD: `lambda_ce_ext` x CE(nhan that hoac nhan gia) + `lambda_kd` x KD(T) voi xac suat cua thay. Khong co nhan dau vao
  (chi audio) -> dung che do ma demo dung.
* `best` = UAR TB tren val (MEAD + CREMA-D + RAVDESS), chi nhan epoch co LMD tinh MEAD val <= (1 + STATIC_TOL) x ban dau.

In [ ]:
CK_LAST, CK_BEST, HIST = OUT / 'phaseG_last.pth', OUT / 'phaseG_best.pth', OUT / 'phaseG_history.json'
torch.manual_seed(CFG['seed']); rng = np.random.default_rng(CFG['seed'])
opt = torch.optim.AdamW(br.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
spe = CFG['steps_per_epoch']; total, warm = CFG['epochs'] * spe, CFG['warmup_epochs'] * spe
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
                                          else 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm)))))
POOL = [v['train'] for v in EXT.values()] + ([UNL] if UNL is not None else [])
POOL_W = np.array([len(P['y3']) for P in POOL], np.float64); POOL_W = POOL_W / POOL_W.sum() if len(POOL) else POOL_W
REMAP = torch.full((8,), -1, dtype=torch.long, device=DEVICE); REMAP[KIDX] = torch.arange(len(KEEP), device=DEVICE)
cw = torch.tensor(len(MTR['y3']) / (3 * np.bincount(MTR['y3'], minlength=3).clip(1)), dtype=torch.float32, device=DEVICE)
STATIC0 = EV0_VAL['MEAD']['lmd_static']


def emb_batch(Dset, idx):
    out = np.empty((len(idx), Dset['full'].shape[1]), np.float32)
    for r, i in enumerate(idx):
        g = Dset['row'][i] if 'row' in Dset else i
        a, b = int(Dset['coff'][g]), int(Dset['coff'][g + 1]); K = b - a
        if K > 1 and rng.random() < CFG['crop_p']:
            kmin = max(1, min(K, int(math.ceil(CFG['crop_min_sec'] / max(CHUNK_SEC, 1e-6)))))
            L = rng.integers(kmin, K + 1); s = rng.integers(0, K - L + 1)
            out[r] = EA.pool(Dset['chunks'], Dset['counts'], a + s, a + s + L)
        else:
            out[r] = Dset['full'][i]
    return torch.tensor(out).to(DEVICE)


def lmd_t(p, y):
    return ((p - y).reshape(len(p), 68, 2).pow(2).sum(-1) + 1e-8).sqrt().mean()


def val_score(ev):
    return float(np.mean([r['uar_tro'] for r in ev.values()]))


start, best, history = 0, val_score(EV0_VAL), []
print('Diem val ban dau (UAR TB) = %.3f | LMD tinh MEAD val = %.4f' % (best, STATIC0))
if CK_LAST.exists():
    ck = torch.load(CK_LAST, map_location=DEVICE, weights_only=False)
    br.load_state_dict(ck['branch_state']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
    start, best, history = ck['epoch'] + 1, ck['best'], ck['history']; rng = np.random.default_rng(CFG['seed'] + start)
    print('>> RESUME tu epoch %d' % start)
elif CK_BEST.exists():
    start = CFG['epochs']; print('>> Da co %s -> bo qua (xoa file nay de hoc lai)' % CK_BEST.name)
nb_m = int(CFG['batch_size'] * (CFG['mead_frac'] if len(POOL) else 1.0)); nb_x = CFG['batch_size'] - nb_m
for ep in range(start, CFG['epochs']):
    br.train(); t0 = time.time(); acc = np.zeros(3)
    for st in range(spe):
        # --- MEAD ---
        j = rng.integers(0, len(MTR['y3']), nb_m)
        E = emb_batch(MTR, j)
        has = torch.tensor((rng.random(nb_m) >= CFG['label_dropout']).astype(np.float32), device=DEVICE)
        S, LG, LV = br(E, torch.tensor(MTR['lab'][j]).to(DEVICE), has)
        e = torch.tensor(MTR['emo'][j], device=DEVICE); lv = torch.tensor(MTR['lvl'][j], dtype=torch.float32, device=DEVICE)
        l_m = (CFG['lambda_static'] * lmd_t(S, torch.tensor(MTR['Y'][j] / ZERO_LMD).to(DEVICE))
               + CFG['lambda_cls'] * F.cross_entropy(LG[:, KIDX], REMAP[e], weight=cw, label_smoothing=CFG['label_smooth']))
        nz = e > 0
        if nz.any():
            l_m = l_m + CFG['lambda_lvl'] * (LV[nz] - lv[nz]).abs().mean()
        loss, l_x = l_m, torch.zeros((), device=DEVICE)
        # --- ngoai MEAD: nhan that / nhan gia + KD voi thay ---
        if nb_x > 0:
            src = rng.choice(len(POOL), nb_x, p=POOL_W)
            Es, Y3, PT = [], [], []
            for s_ in np.unique(src):
                P_ = POOL[s_]; k = int((src == s_).sum()); jj = rng.integers(0, len(P_['y3']), k)
                Es.append(emb_batch(P_, jj)); Y3.append(P_['y3'][jj]); PT.append(P_['p3'][jj])
            E = torch.cat(Es); y3 = torch.tensor(np.concatenate(Y3), device=DEVICE); pt = torch.tensor(np.concatenate(PT), device=DEVICE)
            _, LG, _ = br(E, torch.zeros(len(E), 7, device=DEVICE), torch.zeros(len(E), device=DEVICE))
            lg3 = LG[:, KIDX]; T = CFG['kd_T']
            kd = F.kl_div(F.log_softmax(lg3 / T, -1), (pt.clamp_min(1e-6) ** (1 / T)) / (pt.clamp_min(1e-6) ** (1 / T)).sum(1, keepdim=True),
                          reduction='batchmean') * T * T
            l_x = CFG['lambda_ce_ext'] * F.cross_entropy(lg3, y3, label_smoothing=CFG['label_smooth']) + CFG['lambda_kd'] * kd
            loss = loss + l_x
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(br.parameters(), 1.0); opt.step(); sched.step()
        acc += [loss.item(), l_m.item(), l_x.item()]
    ev = evaluate('val'); sc = val_score(ev); ok = ev['MEAD']['lmd_static'] <= (1 + STATIC_TOL) * STATIC0
    is_best = ok and sc > best
    best = sc if is_best else best
    history.append(dict(epoch=ep + 1, loss=acc[0] / spe, mead=acc[1] / spe, ext=acc[2] / spe, val=ev, score=sc, static_ok=ok))
    torch.save(dict(branch_state=br.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), epoch=ep, best=best,
                    history=history), CK_LAST)
    if is_best:
        torch.save(dict(branch_state=br.state_dict(), epoch=ep, score=sc, val=ev), CK_BEST)
    HIST.write_text(json.dumps(history, indent=1, default=float))
    print('%-3d loss %.3f (MEAD %.3f, ngoai %.3f) | val UAR %s | TB %.3f | LMD tinh %.4f%s %s'
          % (ep + 1, acc[0] / spe, acc[1] / spe, acc[2] / spe, ' '.join('%s %.3f' % (k, r['uar_tro']) for k, r in ev.items()),
             sc, ev['MEAD']['lmd_static'], '' if ok else ' (xau qua)', '<-- BEST' if is_best else ''))
if CK_BEST.exists():
    ck = torch.load(CK_BEST, map_location=DEVICE, weights_only=False); br.load_state_dict(ck['branch_state'])
    print('Chon epoch %d (val UAR TB %.3f)' % (ck['epoch'] + 1, ck['score']))
else:
    br.load_state_dict(F2M['branch_state']); print('Khong epoch nao tot hon ban dau -> giu F2 cu')

Diem val ban dau (UAR TB) = 0.977 | LMD tinh MEAD val = 0.0298
>> Da co phaseG_best.pth -> bo qua (xoa file nay de hoc lai)
Chon epoch 1 (val UAR TB 0.971)


## Cell 7 - Danh gia test: THAY (emotion2vec+) / F2 cu / F2 moi (sau khi hoc them)

In [ ]:
EV1_TEST = evaluate('test')
show_eval(EV0_TEST, 'F2 CU (Phase F2):')
show_eval(EV1_TEST, 'F2 MOI (Phase G):')
print('\nThay doi UAR (moi - cu): ' + ' | '.join('%s %+.3f' % (k, EV1_TEST[k]['uar_tro'] - EV0_TEST[k]['uar_tro']) for k in EV1_TEST))
if 'lmd_static' in EV1_TEST.get('MEAD', {}):
    print('LMD bieu cam tinh MEAD test: cu %.4f -> moi %.4f (%+.1f%%)' % (EV0_TEST['MEAD']['lmd_static'], EV1_TEST['MEAD']['lmd_static'],
          100 * (EV1_TEST['MEAD']['lmd_static'] / EV0_TEST['MEAD']['lmd_static'] - 1)))
print('Luu y: emotion2vec+ (thay) co the da gap CREMA-D / RAVDESS khi train -> diem cua thay tren 2 bo nay co the cao hon thuc te.')
REPORT_G = dict(classes=CLS, before=EV0_TEST, after=EV1_TEST, cfg=CFG, sources={k: {p: int(len(v[p]['y3'])) for p in v} for k, v in EXT.items()},
                unlabeled=int(len(UNL['y3'])) if UNL is not None else 0)
(OUT / 'phaseG_evaluation.json').write_text(json.dumps(REPORT_G, indent=1, default=float))
print('Da luu', OUT / 'phaseG_evaluation.json')


F2 CU (Phase F2):
  nguon     mau    UAR tro(F2)  UAR thay     recall tro neutral/angry/happy
  MEAD      865    0.949        0.950        0.93/0.95/0.97 | LMD tinh 0.0292
  crema     717    0.932        0.944        0.91/0.94/0.94 
  ravdess   80     0.979        0.990        1.00/1.00/0.94 

F2 MOI (Phase G):
  nguon     mau    UAR tro(F2)  UAR thay     recall tro neutral/angry/happy
  MEAD      865    0.949        0.950        0.94/0.94/0.97 | LMD tinh 0.0293
  crema     717    0.944        0.944        0.96/0.93/0.94 
  ravdess   80     0.979        0.990        1.00/1.00/0.94 

Thay doi UAR (moi - cu): MEAD +0.000 | crema +0.012 | ravdess +0.000
LMD bieu cam tinh MEAD test: cu 0.0292 -> moi 0.0293 (+0.2%)
Luu y: emotion2vec+ (thay) co the da gap CREMA-D / RAVDESS khi train -> diem cua thay tren 2 bo nay co the cao hon thuc te.
Da luu /content/drive/MyDrive/DAT301m/Data_student/student_a2l/phaseF/phaseG_evaluation.json


## Cell 8 - Xuat `phaseG_final.pth` (demo tu dung file nay neu co, thay cho `phaseF2_final.pth`)

In [ ]:
final = dict(F2M); final['branch_state'] = br.state_dict(); final['phaseG'] = dict(report=REPORT_G, cfg=CFG)
torch.save(final, OUT / 'phaseG_final.pth')
if CK_LAST.exists():
    CK_LAST.unlink()
print('Da luu', OUT / 'phaseG_final.pth', '| dac trung audio ngoai: %.1f MB' % (sum(f.stat().st_size for f in G_DIR.glob('*.npz')) / 1e6))

Da luu /content/drive/MyDrive/DAT301m/Data_student/student_a2l/phaseF/phaseG_final.pth | dac trung audio ngoai: 23.4 MB
